# Chapter 56: Real-Time Multimedia Processing
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menganalisis dan mengimplementasikan pipeline pemrosesan multimedia waktu-nyata berlatensi rendah (*low-latency multimedia pipeline*), memvalidasi dekomposisi latensi end-to-end, mengevaluasi dinamika buffer antrean, mengukur statistik jitter inter-frame, serta menguji kebijakan pembuangan frame (*drop-oldest*) dalam kondisi beban berlebih (*overload*).

### 1. Landasan Matematis: Latensi End-to-End, Antrean, dan Jitter
Total Latensi Pipeline:
$$T_{\text{total}} = T_{\text{capture}} + T_{\text{queue}} + T_{\text{proc}} + T_{\text{render}}$$

Waktu Tunggu Antrean Buffer:
$$T_{\text{queue}} = \frac{Q}{\mu} \times 1000 \quad \text{(ms)}$$

Rasio Pembuangan Frame Teoritis (Drop Rate):
$$\text{Drop Rate} = \max\left(0, \frac{\lambda - \mu}{\lambda}\right)$$

Statistik Jitter Waktu Kedatangan Antar-Frame:
$$\sigma_{\Delta t} = \sqrt{\frac{1}{N-2} \sum_{i=1}^{N-1} (\Delta t_i - \bar{\Delta t})^2}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Rekayasa Sistem NumPy murni.")

In [ ]:
# 3. Validasi Numerik Skenario A & B (Dekomposisi Latensi & Drop Rate)
def calc_pipeline(t_cap, t_proc, q_occ, t_render):
    mu = 1000.0 / t_proc
    t_queue = (q_occ / mu) * 1000.0
    t_total = t_cap + t_proc + t_queue + t_render
    return t_total, t_queue, mu

# Skenario A: Sistem Seimbang (λ = 30 FPS, T_proc = 18 ms -> μ = 55.56 FPS)
tot_a, q_a, mu_a = calc_pipeline(3.5, 18.0, 0.5, 2.0)
print(f"Skenario A: Total = {tot_a:.2f} ms | Queue = {q_a:.2f} ms | Service Rate = {mu_a:.2f} FPS")
np.testing.assert_allclose(tot_a, 32.5, atol=1e-4)
np.testing.assert_allclose(q_a, 9.0, atol=1e-4)
assert tot_a < 33.33, "Memenuhi tenggat waktu 30 FPS"

# Skenario B: Sistem Overload (λ = 30 FPS, T_proc = 50 ms -> μ = 20 FPS, Q = 2)
tot_b, q_b, mu_b = calc_pipeline(3.5, 50.0, 2.0, 2.0)
drop_b = max(0.0, (30.0 - mu_b) / 30.0)
print(f"Skenario B: Total = {tot_b:.2f} ms | Queue = {q_b:.2f} ms | Drop Rate = {drop_b*100:.2f}%")
np.testing.assert_allclose(tot_b, 155.5, atol=1e-4)
np.testing.assert_allclose(q_b, 100.0, atol=1e-4)
np.testing.assert_allclose(drop_b, 1.0 / 3.0, atol=1e-4)
print("VALIDASI LATENSI & ANTREAN: Skenario A dan B lulus sempurna!")

In [ ]:
# 4. Analisis Jitter Inter-Frame & Stabilitas Waktu Nyata
intervals = np.array([33.3, 34.0, 32.5, 33.8, 33.0], dtype=np.float64)
mean_interval = float(np.mean(intervals))
jitter_std = float(np.std(intervals, ddof=1))
max_deviation = float(np.max(np.abs(intervals - mean_interval)))

print(f"Rata-rata Interval Kedatangan: {mean_interval:.2f} ms")
print(f"Statistik Jitter (Std Dev): {jitter_std:.3f} ms")
print(f"Deviasi Maksimum Jitter: {max_deviation:.2f} ms")

assert 33.0 < mean_interval < 34.0
assert jitter_std >= 0.0
print("VALIDASI JITTER: Lulus verifikasi matematis!")

In [ ]:
# 5. Simulasi Buffer Antrean Produsen-Konsumen Terikat (Drop-Oldest)
def simulate_discrete_queue(arr_fps=30.0, proc_ms=50.0, n_frames=120, max_q=2):
    arr_int = 1000.0 / arr_fps
    queue = []
    server_free_at = 0.0
    dropped = 0
    served = 0
    for i in range(n_frames):
        t_arr = i * arr_int
        while queue and server_free_at <= t_arr:
            item_t = queue.pop(0)
            start_s = max(server_free_at, item_t)
            server_free_at = start_s + proc_ms
            served += 1
        if server_free_at <= t_arr and not queue:
            server_free_at = t_arr + proc_ms
            served += 1
        else:
            if len(queue) >= max_q:
                dropped += 1
                queue.pop(0)
            queue.append(t_arr)
    while queue:
        item_t = queue.pop(0)
        start_s = max(server_free_at, item_t)
        server_free_at = start_s + proc_ms
        served += 1
    return dropped / n_frames

emp_drop = simulate_discrete_queue(30.0, 50.0, 120, 2)
print(f"Empirical Drop Rate (120 frames): {emp_drop*100:.2f}%")
assert abs(emp_drop - 0.33) < 0.05
print("SIMULASI BUFFER: Berhasil memverifikasi kebijakan drop-oldest!")

### 6. Rangkuman & Pembahasan Akademik
1. **Batas Waktu Nyata**: Agar video berjalan mulus tanpa lag, total latensi pipa harus lebih kecil dari interval kedatangan frame ($T_{\text{total}} \le 1000 / \lambda$).
2. **Mitigasi Buffer Bloat**: Antrean tak terbatas menyebabkan penumpukan latensi fatal. Penggunaan buffer cincin (*bounded ring buffer*) dengan kebijakan *drop-oldest* menjamin frame yang diproses selalu terkini.
3. **Relevansi ISVWS 2026–2040**: Kontrol deterministik terhadap jitter dan latensi transmisi merupakan fondasi tele-operasi robotik dan navigasi otonom berpusat penglihatan.